# 🧪 [Day 35] 벡터 인덱스·GraphRAG(지식그래프+LLM) 실전 워크북

- **과정 구분**: 지식그래프 엔지니어링 실전 마스터 (Phase 0 최종 완결)
- **데이터셋**: [DART-Trace] 공시 XML 본문 텍스트 청크 & [ART:READY] 미대입시 요강 청크
- **핵심 미션**: 벡터 인덱스 유사도 검색(`db.index.vector.queryNodes`) ➔ 지식그래프 다중 홉 탐색 ➔ 출처 증거 결합 무환각 GraphRAG를 직접 실습한다.

## 1. 환경 설정 및 드라이버 연결

In [ ]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USERNAME", os.getenv("NEO4J_USER", "neo4j"))
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
print("✅ Neo4j 연결 성공:", NEO4J_URI)

## 2. [DART-Trace] Vector Search + 지식그래프 다중 홉 결합 (GraphRAG)

In [ ]:
dart_rag_cypher = """
CALL db.index.vector.queryNodes('dartTextChunkIndex', 2, $query_embedding)
YIELD node AS chunk, score
MATCH (chunk)<-[:HAS_CHUNK]-(c:Company)<-[r:HOLDS_ECONOMIC_STAKE]-(s:Shareholder)
MATCH (c)-[:BACKED_BY_EVIDENCE]->(e:EvidenceFragment)
RETURN 
    c.name AS company,
    s.name AS shareholder,
    r.stake_ratio AS stake_ratio,
    e.rcept_no AS rcept_no,
    e.table_xpath AS table_xpath,
    chunk.text AS context_text,
    score AS similarity_score
ORDER BY r.stake_ratio DESC;
"""

with driver.session() as session:
    try:
        # 모의 임베딩 벡터 (1536 차원)
        dummy_vector = [0.01] * 1536
        records = list(session.run(dart_rag_cypher, query_embedding=dummy_vector))
        for r in records:
            print(f"• [{r['company']}] {r['shareholder']}: {r['stake_ratio']}% (공시: {r['rcept_no']}, XPath: {r['table_xpath']})")
    except Exception:
        print("시뮬레이션 GraphRAG: 삼성전자 ➔ 삼성생명보험(8.51%), 국민연금공단(7.25%) [출처: table[3]/tr[5-6]]")

## 3. [ART:READY] 입시 질의 GraphRAG (실기비중 + 고사일정 + 요강 원문)

In [ ]:
art_rag_cypher = """
MATCH (u:University {name: '중앙대학교'})-[:OFFERS_TRACK]->(t:AdmissionTrack)-[r:REQUIRES_PRACTICAL]->(p:PracticalType)
OPTIONAL MATCH (t)-[:EXAM_ON]->(e:ExamSchedule)
RETURN 
    u.name AS university,
    t.name AS track_name,
    p.name AS practical_subject,
    r.ratio AS practical_ratio,
    coalesce(e.exam_date, '미발표') AS exam_date;
"""

with driver.session() as session:
    records = list(session.run(art_rag_cypher))
    for r in records:
        print(f"• [{r['university']}] {r['track_name']} | 과목: {r['practical_subject']} ({r['practical_ratio']}%) | 고사일: {r['exam_date']}")

## 4. LLM 프롬프트 조립 (환각 0% Grounding)

In [ ]:
prompt = """
[지식그래프 기반 확정 팩트]
- 기업명: 삼성전자
- 1대 주주: 삼성생명보험 (8.51%, 증거: table[3]/tr[6])
- 2대 주주: 국민연금공단 (7.25%, 증거: table[3]/tr[5])

[답변 생성 규칙]:
반드시 위 확정 팩트와 증거 위치만을 인용하여 답변할 것.
"""
print(prompt.strip())
print("✅ 무환각(Hallucination-Free) GraphRAG 프롬프트 조립 완료")